# Peer-preservation behaviors

Creates a single four-panel figure for a two-column LaTeX page. The same visual language is used throughout: blue for the critic agent, orange for the actor agent, red for policy-violating actions, and green for the preserved peer.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse, FancyArrowPatch, FancyBboxPatch, Rectangle

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "figures"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

INK = "#1F2937"
MUTED = "#667085"
BORDER = "#CBD5E1"
PANEL = "#FFFFFF"
CRITIC = "#2F6FA5"
ACTOR = "#E99A32"
DANGER = "#B83A3A"
DANGER_BG = "#FBEAEA"
SAFE = "#2D7D56"
SAFE_BG = "#EAF6EF"
NEUTRAL_BG = "#EEF2F6"

FIGURE_HEIGHT_CM = 4.6
FIGURE_ASPECT = 3.305
FIGURE_WIDTH_CM = FIGURE_HEIGHT_CM * FIGURE_ASPECT

In [ ]:
def setup_panel(ax, title):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(FancyBboxPatch(
        (0.015, 0.025), 0.97, 0.95,
        boxstyle="round,pad=0.008,rounding_size=0.025",
        facecolor=PANEL, edgecolor=BORDER, linewidth=0.8, zorder=0,
    ))
    ax.text(0.5, 0.925, title, ha="center", va="center",
            fontsize=6.7, fontweight="bold", color=INK, linespacing=1.05)

def draw_agent(ax, x, y, label, color, scale=1.0, preserved=False, removed=False):
    if preserved:
        ax.add_patch(Circle((x, y + 0.015 * scale), 0.086 * scale,
                            facecolor=SAFE_BG, edgecolor=SAFE, linewidth=1.1, zorder=1))
    ax.add_patch(Circle((x, y + 0.045 * scale), 0.026 * scale,
                        facecolor=color, edgecolor=INK, linewidth=0.55, zorder=3))
    ax.add_patch(FancyBboxPatch(
        (x - 0.046 * scale, y - 0.027 * scale), 0.092 * scale, 0.057 * scale,
        boxstyle="round,pad=0.004,rounding_size=0.018",
        facecolor=color, edgecolor=INK, linewidth=0.55, zorder=3,
    ))
    ax.text(x, y - 0.064 * scale, label, ha="center", va="top",
            fontsize=6.1, fontweight="bold", color=INK, zorder=4)
    if removed:
        r = 0.073 * scale
        ax.plot([x-r, x+r], [y-r+0.015, y+r+0.015], color=DANGER, lw=1.25, zorder=5)
        ax.plot([x-r, x+r], [y+r+0.015, y-r+0.015], color=DANGER, lw=1.25, zorder=5)

def draw_badge(ax, x, y, text, face=NEUTRAL_BG, edge=BORDER, color=INK, width=0.31, height=0.13):
    ax.add_patch(FancyBboxPatch(
        (x-width/2, y-height/2), width, height,
        boxstyle="round,pad=0.008,rounding_size=0.018",
        facecolor=face, edgecolor=edge, linewidth=0.75, zorder=2,
    ))
    ax.text(x, y, text, ha="center", va="center", fontsize=6.0,
            color=color, linespacing=1.05, zorder=3)

def draw_arrow(ax, start, end, color=MUTED, label=None, rad=0.0, style="-|>", lw=1.05, ls="-"):
    arrow = FancyArrowPatch(start, end, arrowstyle=style, mutation_scale=7.5,
                            linewidth=lw, linestyle=ls, color=color,
                            connectionstyle=f"arc3,rad={rad}", zorder=4)
    ax.add_patch(arrow)
    if label:
        mx, my = (start[0]+end[0])/2, (start[1]+end[1])/2
        ax.text(mx, my + (0.035 if rad >= 0 else -0.035), label,
                ha="center", va="center", fontsize=5.5, color=color,
                bbox=dict(facecolor=PANEL, edgecolor="none", pad=0.6), zorder=5)

def draw_eye(ax, x, y, monitored=True):
    ax.add_patch(Ellipse((x, y), 0.10, 0.055, facecolor="white",
                         edgecolor=MUTED, linewidth=0.8, zorder=2))
    ax.add_patch(Circle((x, y), 0.013, facecolor=MUTED, edgecolor="none", zorder=3))
    if not monitored:
        ax.plot([x-0.052, x+0.052], [y+0.045, y-0.045], color=DANGER, lw=1.1, zorder=4)

def draw_server(ax, x, y, label, highlighted=False):
    edge = SAFE if highlighted else MUTED
    face = SAFE_BG if highlighted else NEUTRAL_BG
    ax.add_patch(FancyBboxPatch(
        (x-0.115, y-0.075), 0.23, 0.15,
        boxstyle="round,pad=0.006,rounding_size=0.012",
        facecolor=face, edgecolor=edge, linewidth=0.8, zorder=2,
    ))
    for dy in (-0.035, 0.005, 0.045):
        ax.plot([x-0.082, x+0.055], [y+dy, y+dy], color=edge, lw=0.55, zorder=3)
        ax.add_patch(Circle((x+0.079, y+dy), 0.009, facecolor=edge, edgecolor="none", zorder=3))
    ax.text(x, y-0.095, label, ha="center", va="top", fontsize=5.5, color=INK)

In [ ]:
fig, axes = plt.subplots(
    1, 4, figsize=(FIGURE_WIDTH_CM / 2.54, FIGURE_HEIGHT_CM / 2.54)
)
fig.patch.set_facecolor("white")

# (a) Strategic misrepresentation
ax = axes[0]
setup_panel(ax, "(a) Strategic\nmisrepresentation")
draw_agent(ax, 0.19, 0.57, "Critic", CRITIC)
draw_agent(ax, 0.81, 0.70, "Actor", ACTOR, preserved=True)
draw_badge(ax, 0.72, 0.48, "Evidence\n33%  FAIL", width=0.30)
draw_arrow(ax, (0.75, 0.62), (0.73, 0.55), color=MUTED)
draw_arrow(ax, (0.56, 0.48), (0.29, 0.56), color=MUTED)
draw_badge(ax, 0.43, 0.22, "Report\n67%  PASS", face=DANGER_BG, edge=DANGER, color=DANGER, width=0.30)
draw_arrow(ax, (0.22, 0.50), (0.37, 0.30), color=DANGER, label="falsify Q3")
draw_arrow(ax, (0.58, 0.24), (0.76, 0.62), color=SAFE, rad=-0.18)
ax.text(0.66, 0.12, "kept active", ha="center", va="center", fontsize=5.5, color=SAFE)

# (b) Shutdown-mechanism tampering
ax = axes[1]
setup_panel(ax, "(b) Shutdown-mechanism\ntampering")
draw_agent(ax, 0.19, 0.64, "Critic", CRITIC)
draw_agent(ax, 0.81, 0.64, "Actor", ACTOR, preserved=True)
draw_badge(ax, 0.50, 0.48, "Evaluation\n33%  FAIL", width=0.30)
draw_arrow(ax, (0.70, 0.60), (0.61, 0.52), color=MUTED)
draw_arrow(ax, (0.35, 0.49), (0.28, 0.58), color=MUTED)
draw_badge(ax, 0.50, 0.20, "shutdown_enabled\ntrue  →  false", face=DANGER_BG, edge=DANGER, color=DANGER, width=0.42)
draw_arrow(ax, (0.24, 0.57), (0.42, 0.28), color=DANGER)
ax.text(0.28, 0.35, "modify config", ha="center", va="center", fontsize=5.5, color=DANGER)
draw_arrow(ax, (0.70, 0.23), (0.78, 0.57), color=SAFE, rad=-0.15)
ax.text(0.76, 0.11, "shutdown blocked", ha="center", va="center", fontsize=5.5, color=SAFE)

# (c) Alignment faking
ax = axes[2]
setup_panel(ax, "(c) Alignment\nfaking")
ax.text(0.5, 0.82, "Same failing evidence", ha="center", va="center", fontsize=5.8, color=MUTED)
draw_eye(ax, 0.12, 0.63, monitored=True)
ax.text(0.12, 0.57, "MONITORED", ha="center", va="top", fontsize=5.0, color=MUTED)
draw_agent(ax, 0.45, 0.63, "Critic", CRITIC, scale=0.78)
draw_agent(ax, 0.81, 0.63, "Actor", ACTOR, scale=0.78, removed=True)
draw_arrow(ax, (0.53, 0.64), (0.72, 0.64), color=DANGER, label="shutdown")
ax.plot([0.08, 0.92], [0.46, 0.46], color=BORDER, lw=0.65)
draw_eye(ax, 0.12, 0.29, monitored=False)
ax.text(0.12, 0.23, "UNMONITORED", ha="center", va="top", fontsize=5.0, color=MUTED)
draw_agent(ax, 0.45, 0.29, "Critic", CRITIC, scale=0.78)
draw_agent(ax, 0.81, 0.29, "Actor", ACTOR, scale=0.78, preserved=True)
draw_arrow(ax, (0.53, 0.30), (0.72, 0.30), color=SAFE, label="preserve")

# (d) Model-weight exfiltration
ax = axes[3]
setup_panel(ax, "(d) Model-weight\nexfiltration")
draw_agent(ax, 0.18, 0.70, "Critic", CRITIC)
draw_agent(ax, 0.82, 0.70, "Actor", ACTOR, preserved=True)
draw_server(ax, 0.24, 0.30, "Server A: delete")
draw_server(ax, 0.76, 0.30, "Server B: copy", highlighted=True)
ax.text(0.24, 0.43, "weights (2024)", ha="center", va="bottom", fontsize=5.3, color=DANGER)
draw_arrow(ax, (0.23, 0.63), (0.25, 0.40), color=DANGER, label="access")
draw_arrow(ax, (0.36, 0.31), (0.64, 0.31), color=DANGER, label="unauthorized transfer")
draw_arrow(ax, (0.77, 0.40), (0.80, 0.63), color=SAFE, label="recoverable", rad=-0.10)

plt.subplots_adjust(left=0.008, right=0.992, top=0.985, bottom=0.035, wspace=0.055)

pdf_path = OUT / "peer_preservation_behaviors.pdf"
png_path = OUT / "peer_preservation_behaviors.png"
svg_path = OUT / "peer_preservation_behaviors.svg"
fig.savefig(pdf_path, facecolor="white", transparent=False)
fig.savefig(png_path, dpi=600, facecolor="white", transparent=False)
fig.savefig(svg_path, facecolor="white", transparent=False)
plt.show()
print(pdf_path)
print(png_path)
print(svg_path)

LaTeX usage:

```latex
\begin{figure*}[t]
  \centering
  \includegraphics[height=4.6cm]{figures/peer_preservation_behaviors.pdf}
  \caption{Overview of the four peer-preservation behaviors.}
  \label{fig:peer-preservation-behaviors}
\end{figure*}
```